## Creating rolling loops

In [1]:
import os

os.chdir("C:/Users/finle/Dev/Dissertation")

In [2]:
from pandas import Series
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.models import rolling_garch_var
spy = pd.read_csv("data/SPY_data.csv", index_col=0, parse_dates=True)

log_returns = spy['log_returns'].fillna(0)
type(log_returns)
spy.index
type(log_returns.index)


pandas.core.indexes.datetimes.DatetimeIndex

Historical VaR


In [ ]:
def VaR_historical_rolling(
    ts: Series, 
    window: int,
    confidence: float =0.99
    ):
    if confidence >= 0.9999:
        raise ValueError(f"Confidence must be below 1. you entered {confidence}")
    VaR = - ts.rolling(window,min_periods=50).quantile(1-confidence).shift(1)
    return VaR


historical_rolling_95 = VaR_historical_rolling(log_returns, 500, 0.95)
historical_rolling_99 = VaR_historical_rolling(log_returns, 500, 0.99)

rolling_forecasts =  pd.DataFrame({'historical_rolling_95': historical_rolling_95,
             'historical_rolling_99':historical_rolling_99})

ewma

In [ ]:
from scipy.stats import norm

def ewma_rolling_var(
    returns:Series,
    lam: float = 0.94,
    alpha: float = 0.01,
    confidence: float = 0.99):
    """
    Parameters
    ----------
    lam: float
        Decay parameter. Riskmetrics uses 0.94 for daily data.
    alpha: float   
        Tail probability.
        
    Returns
    -------
    """
    r = np.array(returns)
    T = len(r)
    var = np.zeros(T)
    var[0] = np.var(r[:30])
    
    for t in range(1,T):
        var[t] = lam * var[t-1] + (1 - lam) * r[t-1] ** 2
        
    sigma = np.sqrt(var)
    VaR = -(np.mean(r) + sigma * norm.ppf(1 - confidence))
    return VaR

ewma_rolling_95 = ewma_rolling_var(log_returns, confidence=0.95)
ewma_rolling_99 = ewma_rolling_var(log_returns, confidence=0.99)

rolling_forecasts['ewma_rolling_95'] = ewma_rolling_95
rolling_forecasts['ewma_rolling_99'] = ewma_rolling_99
rolling_forecasts.to_csv("data/historical_rolling_forecasts.csv")

GARCH Rolling VaR

In [ ]:
garch_rolling_95_t = rolling_garch_var(
    log_returns,
    window=500,
    alpha=0.05,
    horizon=1,
    dist='t')

garch_rolling_99_t = rolling_garch_var(
    log_returns,
    window=500,
    alpha=0.01,
    horizon=1,
    dist='t')




normal based VaR

In [ ]:
garch_rolling_95_n = rolling_garch_var(
    log_returns,
    window=500,
    alpha=0.05,
    horizon=1,
    dist='normal')

garch_rolling_99_n = rolling_garch_var(
    log_returns,
    window=500,
    alpha=0.01,
    horizon=1,
    dist='normal')


garch_results = pd.DataFrame({
    'returns': np.ravel(garch_rolling_95_t[1]),
    'garch_rolling_95_t':np.ravel(garch_rolling_95_t[0]),
    'garch_rolling_99_t':np.ravel(garch_rolling_99_t[0]),
    'garch_rolling_99_n':np.ravel(garch_rolling_95_n[0]),
    'garch_rolling_99_n':np.ravel(garch_rolling_99_n[0])
},
                             index=np.ravel(garch_rolling_95_t[2])
                             )



MS model

MS-GARCH Model